# Hypothesis Generation with DSPy ReAct + Constrained Semantic Search

This notebook shows how to use DSPy's ReAct agent to generate research hypotheses by searching
across three scientific domains — **neuroscience**, **neuromorphic computing**, and **AI/ML** —
using constrained semantic search.

The dataset contains:
- **750** neuroscience papers  (13 703 sections)
- **1 745** neuromorphic papers (17 390 sections)
- **1 163** AI/ML papers        (23 460 sections)

## What you'll learn
1. Build a semantic index (TF-IDF + sentence embeddings) over multi-domain paper sections
2. Wrap domain-constrained retrieval as DSPy `Tool` functions
3. Wire the tools into a `dspy.ReAct` agent with a hypothesis-generation signature
4. Run the agent and inspect its reasoning trajectory

## 1. Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import dspy

DATA_DIR = Path('../data')

print(f'DSPy {dspy.__version__}')
print(f'Data dir exists: {DATA_DIR.exists()}')

DSPy 3.1.3
Data dir exists: True


## 2. Configure the LLM

Uses the vLLM endpoint at `earlsinclair.ornl.gov:8200` serving `gpt-oss-120b`.

In [2]:
LLM_MODEL       = "openai/gpt-oss-120b"                    # model name as served by vLLM
OLLAMA_BASE_URL = "http://earlsinclair.ornl.gov:8200/v1"   # vLLM endpoint
OLLAMA_API_KEY  = "vllm"

# litellm strips the first "openai/" prefix when building the request body, so
# double-prefix is required to preserve "openai/gpt-oss-120b" in the API call.
_LITELLM_MODEL = f"openai/{LLM_MODEL}"

lm = dspy.LM(
    model=_LITELLM_MODEL,
    api_base=OLLAMA_BASE_URL,
    api_key=OLLAMA_API_KEY,
    max_tokens=16384,
    cache=False,
)

dspy.configure(lm=lm)
print(f"LM configured: {LLM_MODEL} @ {OLLAMA_BASE_URL}")

LM configured: openai/gpt-oss-120b @ http://earlsinclair.ornl.gov:8200/v1


## 3. Load the paper store

`NeuroGraphStore` loads all paper metadata, section chunks, equations,
and the citation graph for each domain.

In [3]:
from storage import NeuroGraphStore

store = NeuroGraphStore.load(DATA_DIR)

print('Domains loaded:', list(store.citation_graphs.keys()))
print('Total sections:', len(store.section_index))
print('Total papers:  ', len(store.paper_index))

Domains loaded: ['aiml', 'neuroscience', 'neuromorphic']
Total sections: 51519
Total papers:   3658


## 4. Build a semantic index

The built-in `search_sections` uses keyword frequency. Here we build a **TF-IDF** index
for fast, approximate semantic retrieval. For heavier workloads you could swap in
`sentence-transformers` embeddings (shown in the optional cell below).

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from storage import SectionChunk

# Build per-domain TF-IDF indices so domain constraints are enforced at index level
DOMAINS = ['neuroscience', 'neuromorphic', 'aiml']

domain_indices: dict[str, tuple[TfidfVectorizer, np.ndarray, list[SectionChunk]]] = {}

for domain in DOMAINS:
    chunks = [c for c in store.section_index.values() if c.domain == domain]
    texts  = [f"{c.heading} {c.body[:600]}" for c in chunks]  # heading-boosted
    
    vec = TfidfVectorizer(max_features=25_000, sublinear_tf=True, stop_words='english')
    matrix = vec.fit_transform(texts)
    
    domain_indices[domain] = (vec, matrix, chunks)
    print(f'{domain:>15s}: {len(chunks):6d} chunks indexed')

print('\nTF-IDF index ready.')

   neuroscience:  12712 chunks indexed
   neuromorphic:  16619 chunks indexed
           aiml:  22188 chunks indexed

TF-IDF index ready.


In [5]:
def semantic_search(query: str, domain: str, top_k: int = 5) -> list[SectionChunk]:
    """Return the top_k most relevant SectionChunks from `domain` for `query`."""
    if domain not in domain_indices:
        raise ValueError(f"Unknown domain '{domain}'. Choose from: {DOMAINS}")
    vec, matrix, chunks = domain_indices[domain]
    q_vec = vec.transform([query])
    scores = cosine_similarity(q_vec, matrix)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [chunks[i] for i in top_idx if scores[i] > 0]


# ── Sanity check: a query that paraphrases rather than quotes the literature ──
query_paraphrased = "neurons suppressing their neighbours so only the strongest signal survives"
query_exact = "lateral inhibition winner-take-all"  # exact terms → TF-IDF works fine

print("TF-IDF results for paraphrased query:")
print(f'  "{query_paraphrased[:70]}…"\n')
results = semantic_search(query_paraphrased, 'neuroscience', top_k=3)
if results:
    for r in results:
        print(f'  [{r.domain}] "{r.title[:60]}" — {r.heading}')
else:
    print("  (no results — TF-IDF cannot match paraphrases)")

print("\nTF-IDF results for exact-term query:")
print(f'  "{query_exact}"\n')
results = semantic_search(query_exact, 'neuroscience', top_k=3)
for r in results:
    print(f'  [{r.domain}] "{r.title[:60]}" — {r.heading}')

TF-IDF results for paraphrased query:
  "neurons suppressing their neighbours so only the strongest signal surv…"

  [neuroscience] "Optimization of graph construction can significantly increas" — 3.2.4 Strongest edges
  [neuroscience] "Mouse higher visual areas provide both distributed and discr" — Role of mouse visual areas in go/no-go speed increment detection task
  [neuroscience] "What is a cognitive map? Organising knowledge for flexible b" — **Inferential planning**

TF-IDF results for exact-term query:
  "lateral inhibition winner-take-all"

  [neuroscience] "A Simple Neural Network Exhibiting Selective Activation of N" — 5 Conclusion
  [neuroscience] "Cell Assembly Dynamics of Sparsely-connected Inhibitory Netw" — The role of lateral inhibition
  [neuroscience] "Thalamocortical network connectivity controls spatiotemporal" — **Lateral thalamic inhibition and thalamocortical delay reshape spatiotemporal cortical dynamics**


### Sentence-transformer embeddings (slower but higher quality)

In [6]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer('all-MiniLM-L6-v2')

dense_indices: dict[str, tuple[np.ndarray, list[SectionChunk]]] = {}
for domain in DOMAINS:
    chunks = [c for c in store.section_index.values() if c.domain == domain]
    texts  = [f"{c.heading} {c.body[:400]}" for c in chunks]
    embeddings = encoder.encode(texts, batch_size=256, show_progress_bar=True,
                                convert_to_numpy=True, normalize_embeddings=True)
    dense_indices[domain] = (embeddings, chunks)
    print(f'{domain}: {len(chunks)} chunks encoded')

def dense_search(query: str, domain: str, top_k: int = 5) -> list[SectionChunk]:
    """Semantic search using sentence-transformer embeddings."""
    if domain not in dense_indices:
        raise ValueError(f"Unknown domain '{domain}'. Choose from: {DOMAINS}")
    emb, chunks = dense_indices[domain]
    q = encoder.encode([query], normalize_embeddings=True)
    scores = cosine_similarity(q, emb)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [chunks[i] for i in top_idx]

# ── Side-by-side comparison on the paraphrased query ──────────────────────────
query_paraphrased = "neurons suppressing their neighbours so only the strongest signal survives"

print(f'\nQuery: "{query_paraphrased}"\n')
print("── TF-IDF ──")
tfidf_hits = semantic_search(query_paraphrased, 'neuroscience', top_k=3)
if tfidf_hits:
    for r in tfidf_hits:
        print(f'  "{r.title[:65]}" — {r.heading}')
else:
    print("  (no results)")

print("\n── Dense (all-MiniLM-L6-v2) ──")
dense_hits = dense_search(query_paraphrased, 'neuroscience', top_k=3)
for r in dense_hits:
    print(f'  "{r.title[:65]}" — {r.heading}')

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

neuroscience: 12712 chunks encoded


Batches:   0%|          | 0/65 [00:00<?, ?it/s]

neuromorphic: 16619 chunks encoded


Batches:   0%|          | 0/87 [00:00<?, ?it/s]

aiml: 22188 chunks encoded

Query: "neurons suppressing their neighbours so only the strongest signal survives"

── TF-IDF ──
  "Optimization of graph construction can significantly increase the" — 3.2.4 Strongest edges
  "Mouse higher visual areas provide both distributed and discrete c" — Role of mouse visual areas in go/no-go speed increment detection task
  "What is a cognitive map? Organising knowledge for flexible behavi" — **Inferential planning**

── Dense (all-MiniLM-L6-v2) ──
  "Cell Assembly Dynamics of Sparsely-connected Inhibitory Networks:" — The role of lateral inhibition
  "Distinct Microcircuit Response to Comparable Input from a Full an" — **INTRODUCTION**
  "Nonlinear transient amplification in recurrent neural networks wi" — **Nonlinear amplification of inputs above a critical threshold**


## 5. Define DSPy Tool functions

Each tool constrains the search to a specific domain, giving the agent clear boundaries
while allowing cross-domain reasoning at the hypothesis level.

In [7]:
def _format_chunks(chunks: list[SectionChunk], max_body: int = 400) -> str:
    """Render retrieved chunks into a compact string for the agent context."""
    if not chunks:
        return 'No relevant results found.'
    lines = []
    for i, c in enumerate(chunks, 1):
        body_snippet = c.body[:max_body].replace('\n', ' ').strip()
        lines.append(
            f"[{i}] {c.title[:70]}\n"
            f"    Section: {c.heading}\n"
            f"    Excerpt: {body_snippet}\u2026\n"
            f"    Source: {c.domain}/{c.paper_id} (doi:{c.doi})"
        )
    return '\n\n'.join(lines)


def search_papers(query: str, domain: str = "") -> str:
    """Search scientific papers by semantic query, optionally constrained to one domain.

    Args:
        query:  Natural-language search query (paraphrases work — dense embeddings are used).
        domain: Optional domain filter. One of 'neuroscience', 'neuromorphic', 'aiml'.
                Leave empty to search all domains simultaneously.
    """
    if domain and domain not in DOMAINS:
        return f"Unknown domain '{domain}'. Choose from: {DOMAINS} or leave empty for all."

    if domain:
        return _format_chunks(dense_search(query, domain, top_k=5))

    all_chunks: list[SectionChunk] = []
    for d in DOMAINS:
        all_chunks.extend(dense_search(query, d, top_k=3))
    return _format_chunks(all_chunks)


def find_cross_domain_links(concept: str = "") -> str:
    """Search for a concept across all three domains to find cross-disciplinary connections.
    Use this when you want to see how a concept bridges neuroscience, neuromorphic, and AI/ML.

    Args:
        concept: the concept to search for (e.g. 'lateral inhibition', 'attention mechanism')
    """
    lines = [f"Cross-domain search for '{concept}':\n"]
    found_in = []
    for dom in DOMAINS:
        chunks = dense_search(concept, dom, top_k=3)
        if chunks:
            found_in.append(dom)
            lines.append(f"[{dom}] — {len(chunks)} matches")
            lines.append(_format_chunks(chunks, max_body=200))
            lines.append("")

    if not found_in:
        return f"Concept '{concept}' not found in any domain."

    if len(found_in) >= 3:
        lines.append("STRONG BRIDGE: concept spans all three domains!")
    elif len(found_in) == 2:
        lines.append(f"PARTIAL BRIDGE: found in {' + '.join(found_in)}")

    return "\n".join(lines)


def get_cited_by(domain: str, paper_id: str) -> str:
    """Return papers that cite a given paper, useful for tracing the impact of an idea.

    Args:
        domain:   One of 'neuroscience', 'neuromorphic', 'aiml'.
        paper_id: Numeric paper ID (e.g. '42').
    """
    neighbors = store.get_citation_neighbors(domain, paper_id)
    cited_by = neighbors['cited_by'][:8]
    if not cited_by:
        return f'No citing papers found for {domain}/{paper_id}.'
    lines = [f'Papers citing {domain}/{paper_id}:']
    for pid in cited_by:
        node = store.paper_index.get(f'{domain}/{pid}')
        title = node.title[:70] if node else '(unknown)'
        lines.append(f'  {domain}/{pid}: {title}')
    return '\n'.join(lines)


TOOLS = [search_papers, find_cross_domain_links, get_cited_by]
print('Tools defined:', [t.__name__ for t in TOOLS])

Tools defined: ['search_papers', 'find_cross_domain_links', 'get_cited_by']


## 6. Define the DSPy Signature

The signature tells the agent what its goal is and what form the output should take.
Adding field descriptions shapes how the agent searches and synthesises.

In [8]:
class HypothesisGen(dspy.Signature):
    """You are a cross-disciplinary research scientist.

    Given a research question, use the available tools to gather evidence across
    neuroscience, neuromorphic computing, and AI/ML literature, then synthesise
    a novel, testable hypothesis.

    Available tools:
    - search_papers(query, domain="") — semantic search over paper sections.
      Pass domain="neuroscience", "neuromorphic", or "aiml" to constrain the search,
      or omit domain (or pass "") to search all three at once.
    - get_cited_by(domain, paper_id) — find papers that cite a given paper.

    Strategy: start broad with domain="" to get a cross-domain overview, then
    drill into specific domains or papers for deeper evidence before synthesising.
    """

    question: str = dspy.InputField(
        desc="The research question or problem area to generate hypotheses for."
    )

    hypothesis: str = dspy.OutputField(
        desc="A specific, testable hypothesis grounded in evidence from multiple domains. "
             "Format: 'If [mechanism from domain A] is combined with [approach from domain B], "
             "then [predicted outcome] because [evidence chain].' Cite paper IDs."
    )

    evidence_summary: str = dspy.OutputField(
        desc="Bullet-point summary of the 3-5 most relevant findings across domains "
             "that support the hypothesis, with domain and paper ID citations."
    )

    open_questions: str = dspy.OutputField(
        desc="2-3 open questions or experiments that would validate or refute the hypothesis."
    )

print('Signature defined.')

Signature defined.


## 7. Build the ReAct agent

In [9]:
agent = dspy.ReAct(
    signature=HypothesisGen,
    tools=TOOLS,
    max_iters=12,   # enough for 2-3 searches per domain
)

print('ReAct agent ready.')
print(f'Max iterations: {agent.max_iters}')
print(f'Tools: {list(agent.tools.keys())}')

ReAct agent ready.
Max iterations: 12
Tools: ['search_papers', 'find_cross_domain_links', 'get_cited_by', 'finish']


## 8. Run the agent

Ask a question that bridges neuroscience and neuromorphic engineering.

In [10]:
QUESTION = (
    "How could cortical lateral inhibition mechanisms found in neuroscience "
    "be implemented in spiking neural networks to improve sparse, energy-efficient "
    "object recognition in neuromorphic hardware?"
)

result = agent(question=QUESTION)

print('='*70)
print('QUESTION')
print('='*70)
print(QUESTION)
print()

QUESTION
How could cortical lateral inhibition mechanisms found in neuroscience be implemented in spiking neural networks to improve sparse, energy-efficient object recognition in neuromorphic hardware?



## 9. Inspect results

In [11]:
print('='*70)
print('HYPOTHESIS')
print('='*70)
print(result.hypothesis)
print()
print('='*70)
print('EVIDENCE SUMMARY')
print('='*70)
print(result.evidence_summary)
print()
print('='*70)
print('OPEN QUESTIONS')
print('='*70)
print(result.open_questions)

HYPOTHESIS
**If** cortical‑inspired lateral inhibition implemented as conductance‑based divisive normalization between neighboring excitatory spiking neurons **is combined with** a spiking winner‑take‑all (1‑spike‑WTA) layer that enforces hard competition across each feature map, **then** a spiking convolutional network deployed on neuromorphic hardware will achieve higher object‑recognition accuracy while reducing total spike count by ≥40 % and lowering energy consumption by ≥30 % relative to a comparable dense SNN, **because** (1) lateral inhibition decorrelates redundant feature responses and creates sparse coding in V1‑like layers【neuroscience/8】, (2) adding a lateral‑inhibition stage improves classification separability on neuromorphic chips【neuromorphic/1292】, and (3) WTA inhibition forces a minimal active subpopulation, cutting spike traffic and power on event‑driven processors【aiml/561】.

EVIDENCE SUMMARY
- **Neuroscience (cortical lateral inhibition)** – Cell‑assembly dynamics

## 10. Inspect the reasoning trajectory

DSPy ReAct stores every thought, tool call, and observation in `result.trajectory`.
This is the agent's full chain of reasoning.

In [12]:
traj = result.trajectory
n_steps = sum(1 for k in traj if k.startswith('thought_'))
print(f'Agent took {n_steps} reasoning steps\n')

for i in range(n_steps):
    thought   = traj.get(f'thought_{i}', '')
    tool_name = traj.get(f'tool_name_{i}', '')
    tool_args = traj.get(f'tool_args_{i}', {})
    obs       = traj.get(f'observation_{i}', '')

    print(f'─── Step {i+1} ───────────────────────────────')
    print(f'Thought:   {thought}')
    print(f'Tool:      {tool_name}({tool_args})')
    if tool_name != 'finish':
        # Truncate long observations
        obs_preview = str(obs)[:300] + ('…' if len(str(obs)) > 300 else '')
        print(f'Observed:  {obs_preview}')
    print()

Agent took 12 reasoning steps

─── Step 1 ───────────────────────────────
Thought:   To build a solid, cross‑disciplinary foundation for the hypothesis, I need to understand how the concept of **lateral inhibition** is discussed in each of the three relevant domains: (1) cortical neuroscience (mechanistic descriptions, role in sparse coding), (2) neuromorphic engineering (hardware primitives, existing spiking implementations), and (3) AI/ML (spiking neural network architectures, object recognition, energy‑efficiency). Using the `find_cross_domain_links` tool will surface key papers or review sections that explicitly connect lateral inhibition across these areas, giving me a map of the existing evidence and any gaps. This will guide subsequent, more focused searches within each domain.
Tool:      find_cross_domain_links({'concept': 'lateral inhibition'})
Observed:  Cross-domain search for 'lateral inhibition':

[neuroscience] — 3 matches
[1] The mechanism of directionally selective unit

## 11. Try a different question

Change the question and re-run cells 8–10. Here are some ideas:

```python
# STDP + deep learning
QUESTION = (
    "Can spike-timing-dependent plasticity (STDP) from neuroscience "
    "replace backpropagation for training deep spiking networks?"
)

# Temporal coding
QUESTION = (
    "How does temporal spike coding in biological neurons compare to "
    "rate coding, and what are the implications for neuromorphic classification accuracy?"
)

# Energy efficiency
QUESTION = (
    "What biological principles of synaptic pruning could reduce "
    "the memory footprint of spiking neural networks on neuromorphic chips?"
)
```

In [13]:
# Batch comparison: run the agent on multiple questions and compare hypotheses
questions = [
    "How could STDP be used to train recurrent spiking networks for sequence recognition?",
    "What role does neural oscillation play in cortical computation and how could it be exploited in SNNs?",
]

hypotheses = []
for q in questions:
    r = agent(question=q)
    hypotheses.append({'question': q, 'hypothesis': r.hypothesis})
    print(f'Q: {q[:60]}...')
    print(f'H: {r.hypothesis[:200]}...')
    print()

Q: How could STDP be used to train recurrent spiking networks f...
H: **If** a recurrent spiking network is equipped with a surrogate‑gradient learning rule that mimics the temporal window and eligibility‑trace dynamics of biological STDP (domain A: neuroscience), **and...

Q: What role does neural oscillation play in cortical computati...
H: **If cortical‑inspired communication‑through‑coherence (phase‑aligned excitatory–inhibitory oscillations) is combined with resonator‑and‑fire neurons that fire preferentially at a specific phase of th...

